In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from IPython.display import HTML, Markdown
import xarray as xr
import plotly.graph_objects as go
import seaborn as sns

import nn_gmm as nng
import ml_tools as mlt

In [ ]:
hp_result_dir = Path("/Users/claudy/dev/work/data/nn_gmm/hp_opt/0121_1411")
db_ffp = hp_result_dir / "0121_1411.db"

In [ ]:
study_id = hp_result_dir.name

study = optuna.load_study(
    storage=f"sqlite:///{hp_result_dir}/{study_id}.db"
    , study_name=study_id
)

In [ ]:
df = study.trials_dataframe()
df = df.loc[df.state == "COMPLETE"]
df["trial_id"] = [f"trial_{row['number']:03d}" for _, row in df.iterrows()]
df = df.sort_values("value")

In [ ]:
result_cols = ["number",'value', 'duration',
       'params_activation_fn', 'params_batch_size', 'params_dropout_rate',
       'params_l2_reg', 'params_learning_rate', 'params_n_layers',
       'params_unit_size', 'params_last_hlayer_size', 'params_use_batch_norm']
param_cols = [col for col in df.columns if col.startswith("params_")]

In [ ]:
HTML(df[result_cols].head(25).to_html())

In [ ]:
# Group by parameters and get mean value to address duplicates
Markdown(df.groupby(param_cols)["value"].mean().sort_values().head(10).to_frame().to_markdown())

## Loss Investigation


In [ ]:
# Load the metrics for every trial
trial_dirs = list(hp_result_dir.glob("trial_*"))
trial_names = [cur_dir.name for cur_dir in trial_dirs]

In [ ]:
metrics_list = [xr.load_dataarray(cur_dir / "metrics.nc") for cur_dir in trial_dirs]

In [ ]:
metrics_da = xr.concat(metrics_list, dim="trial")
metrics_da = metrics_da.assign_coords({"trial": trial_names})

In [ ]:
n_trials = 15

top_x_trials = df.head(n_trials)["trial_id"].values
# line_objects = [go.Scatter(x=metrics_da.coords["epoch"].values, y=metrics_da.sel(trial=trial, metric="w_loss_hist_val").median(dim="cv_iter"), mode="lines", name=trial) for trial in top_10_trials]
# fig = go.Figure(data=line_objects)

fig = go.Figure()
fig.update_layout(width=1600, height=800,
                  margin=dict(l=10, r=10, t=30, b=10)  # left, right, top, bottom
)
fig.update_yaxes(range=[-2.75, -1.5]) 

colors = sns.color_palette("tab10", n_colors=n_trials).as_hex()
for cur_trial, cur_color in zip(top_x_trials, colors):
    cur_16th = metrics_da.sel(trial=cur_trial, metric="w_loss_hist_val").quantile(0.16, dim="cv_iter")
    cur_84th = metrics_da.sel(trial=cur_trial, metric="w_loss_hist_val").quantile(0.84, dim="cv_iter")
    cur_std = metrics_da.sel(trial=cur_trial, metric="w_loss_hist_val").std(dim="cv_iter")
    

    fig.add_trace(go.Scatter(x=metrics_da.coords["epoch"].values, y=metrics_da.sel(trial=cur_trial, metric="w_loss_hist_val").median(dim="cv_iter"), mode="lines", 
                             name=cur_trial, legendgroup=cur_trial, line=dict(color=cur_color)))
    fig.add_trace(go.Scatter(x=metrics_da.coords["epoch"].values, y=cur_16th, mode="lines", line=dict(dash="dash", color=cur_color), showlegend=False, legendgroup=cur_trial))
    fig.add_trace(go.Scatter(x=metrics_da.coords["epoch"].values, y=cur_84th, mode="lines", line=dict(dash="dash", color=cur_color), showlegend=False, legendgroup=cur_trial))

fig.show()